In [8]:
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

from backend.extract_filter import classify_and_extract
from backend.data_utils import fetch_papers, verify_results, chunk_papers, format_query, clear
from backend.Database import create_index, save_embeddings_to_index, delete_index
from backend.retrieval import answer_query, clear_memory
from backend.pdf_ingest import ingest_uploaded_pdf
from pinecone import Pinecone
pc = Pinecone(api_key=os.getenv("PINECONE_API_KEY"))


from time import sleep

In [9]:
# from pinecone import Pinecone

# indexes = pc.list_indexes()

# # delete all indexes
# for idx in indexes:
#     index_name = idx["name"]
#     pc.delete_index(index_name)
#     print(f"Deleted index: {index_name}")


In [10]:
# Create the index once (assuming it should persist during the conversation)
index_name = create_index()
index_obj = pc.Index(index_name)  

Index '4i6thfhob4' created successfully!


In [ ]:
clear_memory()
clear()

def handleUserQuery(user_input, user_index):

    # path = "Hacohen2022.pdf"
    # ingest_uploaded_pdf(file_path=path, user_index=index_name)

    # Use the provided user_input instead of the external variable
    classification = classify_and_extract(user_input)
    print(classification)
 
    filters = classification['filters']
    print("\nRunning Search with filters:", filters)

    # if no academic query present, call llm without fetching paper
    if not format_query(filters):   
        question = classification['question']
        answer = answer_query(classification, index_obj, top_k=100)
        print("Generated Answer:")
        print(answer)
        return

    # Step 1: Fetch papers
    papers = fetch_papers(filters)

    # Step 2: Verify & Filter papers
    verified_papers = verify_results(papers, filters)
    print(f"[integration] Found {len(verified_papers)} verified paper(s).")
    
    # Determine number of papers to return; default to 1 if not provided
    try:
        num_papers = int(filters.get("max_results", 1))
    except ValueError:
        num_papers = 1
    num_papers = min(num_papers, len(verified_papers))

    # Step 3: Chunk papers
    chunks = chunk_papers(verified_papers[:num_papers])    # Get full paper content as chunks
    texts = [chunk["text"] for chunk in chunks]            # Extract just the text
    print("text:")
    print(texts)

    titles = [chunk["title"] for chunk in chunks]    # Extract the titles
    print('titles: ')
    print(titles)

    authors = [[author.lower() for author in chunk["author"]] for chunk in chunks]
    print("author:")
    print(authors)

    dates = [chunk["date"] for chunk in chunks]    # Extract the titles
    print('date: ')
    print(dates)


    if texts:
        # Save the full paper text chunks to the index in batches
        batch_size = 200
        for i in range(0, len(texts), batch_size): 
            print('i: ', i)
            text_batch = texts[i:i+batch_size]
            title_batch = titles[i:i+batch_size]      
            author_batch = authors[i:i+batch_size]    
            date_batch = dates[i:i+batch_size]        
            source_batch = ["arxiv"] * len(text_batch)
            save_embeddings_to_index(
                index_name=user_index, 
                text=text_batch, 
                titles=title_batch, 
                authors=author_batch, 
                dates=date_batch, 
                sources=source_batch,
                namespace="default"
            )
            sleep(1)



    #answer search question
    answer = answer_query(classification=classification, index_obj=index_obj, top_k=100)
    print("Generated Answer:")
    print(answer)
 
# Use a loop to enable continuous conversation
while True: 
    user_query = input("Enter your query (or type 'exit' to quit): ")
    if user_query.lower() in ["exit", "quit"]:
        # delete_index(index_name=index_name)
        clear_memory()
        clear()
        break
    handleUserQuery(user_query, index_name)


Memory cleared.
{'action': 'search', 'filters': {'query': None, 'author': 'jie zhang', 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'newest'}, 'question': 'What is paper title, author and date? Summarise latest papers from jie zhang'}

Running Search with filters: {'query': None, 'author': 'jie zhang', 'title': None, 'category': None, 'abstract': None, 'journal_ref': None, 'doi': None, 'exclude_words': None, 'start_date': None, 'end_date': None, 'max_results': '1', 'sort_by': 'newest'}
added  BoxSeg: Quality-Aware and Peer-Assisted Learning for Box-supervised Instance Segmentation  to prevent searching it next time
[integration] Found 1 verified paper(s).
Extracted 321 chunks from 'BoxSeg: Quality-Aware and Peer-Assisted Learning for Box-supervised Instance Segmentation'

Total Chunks Extracted: 321

text:
['BoxSeg: Quality-Aware and Peer-Assisted Learning

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retrying, relevant documents are found
After retr

WARNING! top_p is not default parameter.
                    top_p was transferred to model_kwargs.
                    Please confirm that top_p is what you intended.
WARNING! frequency_penalty is not default parameter.
                    frequency_penalty was transferred to model_kwargs.
                    Please confirm that frequency_penalty is what you intended.
WARNING! presence_penalty is not default parameter.
                    presence_penalty was transferred to model_kwargs.
                    Please confirm that presence_penalty is what you intended.


Retrieved chunks: 
[{'text': '\'class HelloWorld {     // Broadcast to all known hosts \n    public static void main() { \n        hosts.+print("Hello, World!\\n" +  \n                            this_host.name() + ":-)\\n"); \n    }  \n}; \n \nThe two Hello concepts illustrated by this example are a\'\ntext from The distributed Language Hello White Paper\nauthored by boris burshteyn', 'title': 'the distributed language hello white paper', 'author': ['boris burshteyn'], 'date': 20140914.0, 'source': 'arxiv'}, {'text': "'transferring code, data and control flow between the \nengines from the connected hosts. Hello is a protocol -\nagnostic language, as it imposes no requirements on the \nunderlying network protocol; neither has it exposed any \nof the protocol elements to Hello programs3.'\ntext from The distributed Language Hello White Paper\nauthored by boris burshteyn", 'title': 'the distributed language hello white paper', 'author': ['boris burshteyn'], 'date': 20140914.0, 'source':

In [ ]:
# 